# Masterclass Notebook: Transfer Learning Foundations to Implementation

## Section 1: Theoretical Foundations, Domains, and Tasks

### 1. Mathematical Formalization
Transfer learning aims to improve the learning of a target predictive function $f_t(\cdot)$ in a target domain $D_t$ using knowledge from a source domain $D_s$ and source task $T_s$.

* **Source Domain:** $D_s = \{\mathcal{X}_s, P(X_s)\}$ where $\mathcal{X}_s$ is the feature space and $P(X_s)$ is the marginal probability distribution.
* **Source Task:** $T_s = \{\mathcal{Y}_s, f_s(\cdot)\}$ where $\mathcal{Y}_s$ is the label space and $f_s(\cdot)$ is the predictive function learned from training pairs.
* **Target Domain & Task:** $D_t = \{\mathcal{X}_t, P(X_t)\}$ and $T_t = \{\mathcal{Y}_t, f_t(\cdot)\}$.

Transfer learning is applied when $D_s \neq D_t$ or $T_s \neq T_t$.

### 2. Core Assumptions
1. **Feature Transferability:** Low-level features (edges, textures) are domain-agnostic and highly transferable across tasks.
2. **Data Asymmetry:** Source dataset size $|D_s| \gg |D_t|$ (Target dataset size). Abundant source data prevents target overfitting.
3. **Task/Domain Relatedness:** $D_s$ and $D_t$ must share latent structural similarity to avoid **Negative Transfer**.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

# Set device context
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

# 1. Acquire Pre-trained Source Model (Ds, Ts)
print("[Source Setup] Loading pre-trained ResNet-18 backbone...")
source_weights = models.ResNet18_Weights.DEFAULT
source_model = models.resnet18(weights=source_weights).to(device)

# 2. Setup Target Domain (Dt, Tt) - Simulating Data Asymmetry (|Ds| >> |Dt|)
target_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("[Target Setup] Initializing Target Dataset (CIFAR-10 subset)...")
target_train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=target_transform
)

# Subsampling target data to reflect data asymmetry constraints
small_target_indices = torch.arange(1000)
subset_target_train = torch.utils.data.Subset(target_train_dataset, small_target_indices)
target_loader = DataLoader(subset_target_train, batch_size=32, shuffle=True)
print(f"[Target Setup] Target Dt initialized with {len(subset_target_train)} training samples.")

Executing on device: cuda
[Source Setup] Loading pre-trained ResNet-18 backbone...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s]


[Target Setup] Initializing Target Dataset (CIFAR-10 subset)...


100%|██████████| 170M/170M [24:47<00:00, 115kB/s]


[Target Setup] Target Dt initialized with 1000 training samples.


## Section 2: Network Modification and Visualizing Feature Reuse

To adapt a pre-trained network to target domain $D_t$ and task $T_t$:
1. **Classification Head Replacement:** Replace source output layer $f_s(\cdot)$ ($1000$ classes) with $f_t(\cdot)$ ($10$ classes).
2. **Hierarchy of Representation:**
   - **Early Layers (`conv1`, `layer1`):** Capture low-level features with high transferability.
   - **Deep Layers (`layer4`, `fc`):** Capture high-level task-specific semantic concepts.

In [ ]:
num_target_classes = 10

# Replace source linear layer with target output head
in_features = source_model.fc.in_features
print(f"Original Head Features: {source_model.fc.out_features}")
source_model.fc = nn.Linear(in_features, num_target_classes).to(device)
print(f"Modified Head Features: {source_model.fc.out_features}")

# Feature Transferability Inspection via Forward Hooks
activation_map = {}
def get_activation(name):
    def hook(model, input, output):
        activation_map[name] = output.detach()
    return hook

source_model.layer1[0].conv1.register_forward_hook(get_activation("early_features"))
source_model.layer4[0].conv1.register_forward_hook(get_activation("deep_features"))

dummy_input = torch.randn(1, 3, 224, 224).to(device)
_ = source_model(dummy_input)

print("\n--- Feature Representation Map Shapes ---")
print(f"Early Generic Features (Layer 1): {activation_map['early_features'].shape}")
print(f"Deep Semantic Features  (Layer 4): {activation_map['deep_features'].shape}")

Original Head Features: 1000
Modified Head Features: 10

--- Feature Representation Map Shapes ---
Early Generic Features (Layer 1): torch.Size([1, 64, 56, 56])
Deep Semantic Features  (Layer 4): torch.Size([1, 512, 7, 7])


## Section 3: Adaptation Regimes — Freezing, Fine-Tuning, and PEFT

| Adaptation Strategy | Unfrozen Parameters | Target Data Requirement |
| :--- | :--- | :--- |
| **Linear Probing** | Classifier Head Only | Very Low |
| **Controlled Partial Unfreezing** | Head + Final Layer Block | Low to Medium |
| **Full Fine-Tuning** | Whole Network ($W_{all}$) | Large |
| **PEFT / LoRA** | Injected Low-Rank Modules | Medium |

### Controlled Unfreezing & PEFT Architecture
- **Controlled Unfreezing Exception:** On small datasets with slight domain shift or class imbalance, unfreezing the head + final residual block (`layer4`) allows high-level adjustment without catastrophic forgetting.
- **PEFT / LoRA Concept:** Parameterizes updates as $W = W_0 + B \cdot A$ with rank $r \ll \min(d,k)$, leaving $W_0$ frozen.

In [ ]:
import copy

def set_parameter_requires_grad(model, feature_extracting=True):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True

# Strategy A: Linear Probing
model_linear_probe = copy.deepcopy(source_model)
set_parameter_requires_grad(model_linear_probe, feature_extracting=True)

# Strategy B: Controlled Partial Unfreezing (Layer 4 + FC Head)
model_partial_unfreeze = copy.deepcopy(source_model)
set_parameter_requires_grad(model_partial_unfreeze, feature_extracting=True)
for param in model_partial_unfreeze.layer4.parameters():
    param.requires_grad = True

optimizer_partial = optim.Adam([
    {'params': model_partial_unfreeze.layer4.parameters(), 'lr': 1e-4},
    {'params': model_partial_unfreeze.fc.parameters(), 'lr': 1e-3}
])

# Strategy C: Conceptual LoRA Wrapper
class LoRALinearWrapper(nn.Module):
    def __init__(self, original_linear: nn.Linear, rank: int = 4):
        super().__init__()
        self.original_linear = original_linear
        self.original_linear.weight.requires_grad = False
        if self.original_linear.bias is not None:
            self.original_linear.bias.requires_grad = False
        in_dim = original_linear.in_features
        out_dim = original_linear.out_features
        self.lora_A = nn.Parameter(torch.randn(rank, in_dim) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_dim, rank))
        self.scaling = 1.0 / rank

    def forward(self, x):
        base_out = self.original_linear(x)
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T
        return base_out + (lora_out * self.scaling)

model_peft = copy.deepcopy(source_model)
model_peft.fc = LoRALinearWrapper(model_peft.fc, rank=4).to(device)
print("[Adaptation Regimes Configured] Linear Probe, Controlled Unfreeze, and LoRA instantiated.")

[Adaptation Regimes Configured] Linear Probe, Controlled Unfreeze, and LoRA instantiated.


## Section 4: Task Relatedness and Negative Transfer Diagnostics

### Negative Transfer Dynamics
Negative transfer occurs when pre-trained source knowledge degrades target task accuracy compared to training from scratch.

### Diagnostic Benchmarking Protocol
1. **Scratch Baseline:** Randomly initialized weights ($W_0 \sim \mathcal{N}(0, \sigma^2)$).
2. **Positive Transfer:** Pre-trained ImageNet backbone.
3. **Mismatched Transfer:** Corrupted/distorted feature representations.

In [ ]:
import time

def train_and_evaluate(model, dataloader, criterion, optimizer, epochs=2):
    model.train()
    for epoch in range(epochs):
        running_loss, correct, total = 0.0, 0, 0
        start_time = time.time()
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
            total += inputs.size(0)
        epoch_loss = running_loss / total
        epoch_acc = correct.double() / total
        print(f"   Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Time: {time.time()-start_time:.2f}s")

criterion = nn.CrossEntropyLoss()

print("\n=== Experiment 1: Scratch Model (Random Initialization) ===")
scratch_model = models.resnet18(weights=None)
scratch_model.fc = nn.Linear(scratch_model.fc.in_features, num_target_classes).to(device)
opt_scratch = optim.Adam(scratch_model.parameters(), lr=1e-3)
train_and_evaluate(scratch_model, target_loader, criterion, opt_scratch, epochs=2)

print("\n=== Experiment 2: Positive Transfer (ImageNet Pre-trained) ===")
positive_transfer_model = copy.deepcopy(model_linear_probe)
opt_positive = optim.Adam(positive_transfer_model.fc.parameters(), lr=1e-3)
train_and_evaluate(positive_transfer_model, target_loader, criterion, opt_positive, epochs=2)

print("\n=== Experiment 3: Negative Transfer Risk (Corrupted Pre-trained Weights) ===")
corrupted_model = models.resnet18(weights=None)
with torch.no_grad():
    for param in corrupted_model.parameters():
        param.add_(torch.randn(param.size()) * 100.0)
corrupted_model.fc = nn.Linear(corrupted_model.fc.in_features, num_target_classes).to(device)
set_parameter_requires_grad(corrupted_model, feature_extracting=True)
opt_corrupted = optim.Adam(corrupted_model.fc.parameters(), lr=1e-3)
train_and_evaluate(corrupted_model, target_loader, criterion, opt_corrupted, epochs=2)


=== Experiment 1: Scratch Model (Random Initialization) ===


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same